In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.tree import DecisionTreeClassifier
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay,precision_recall_curve
from sklearn.metrics import roc_curve,roc_auc_score,f1_score
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Load datasets
train_df = pd.read_excel("/content/drive/MyDrive/Final_Thesis/final_d2.xlsx")   # Training dataset
test_df = pd.read_excel("/content/drive/MyDrive/Final_Thesis/final_d3.xlsx")    # Test dataset

print(f"Training dataset size: {train_df.shape[0]}")
print(f"Test dataset size: {test_df.shape[0]}")

# Extract model predictions columns
pred_cols = ["urlnet_prediction", "htmlphish_prediction", "webphish_prediction"]

# Handle missing values in the test set by dropping rows with NaN in prediction columns or true_label
train_df_cleaned = train_df.dropna(subset=pred_cols + ["true_label"])
X_train = train_df_cleaned[pred_cols].values
y_train = train_df_cleaned["true_label"].values

# Handle missing values in the test set by dropping rows with NaN in prediction columns or true_label
test_df_cleaned = test_df.dropna(subset=pred_cols + ["true_label"])
X_test = test_df_cleaned[pred_cols].values
y_test = test_df_cleaned["true_label"].values

print(f"\nTraining features shape: {X_train.shape}")
print(f"Test features shape: {X_test.shape}")

Training dataset size: 16000
Test dataset size: 9373

Training features shape: (16000, 3)
Test features shape: (9373, 3)


In [ ]:
# --- Basic Ensemble Functions ---
def majority_voting(predictions):
    binary_preds = (predictions >= 0.5).astype(int)
    return (binary_preds.sum(axis=1) >= 2).astype(int)

def mean_voting(predictions):
    mean_preds = predictions.mean(axis=1)
    return (mean_preds >= 0.5).astype(int)

def most_certain(predictions):
    # Calculate certainty (distance from 0.5) for each prediction
    certainties = np.abs(predictions - 0.5)
    # Get index of most certain prediction for each sample
    most_certain_idx = np.argmax(certainties, axis=1)

    # Get the most certain predictions
    result = []
    for i, idx in enumerate(most_certain_idx):
        result.append(1 if predictions[i, idx] >= 0.5 else 0)

    return np.array(result)

In [ ]:
# --- Meta-Model Training Functions ---
def train_decision_tree_meta_model(X_train, y_train):
    dt_meta = DecisionTreeClassifier(
        max_depth=5,           # Prevent overfitting
        min_samples_split=10,  # Minimum samples to split
        min_samples_leaf=5,    # Minimum samples in leaf
        random_state=42
    )

    dt_meta.fit(X_train, y_train)

    # Print feature importance
    model_names = ["URLNet", "HTMLPhish", "WebPhish"]
    print(f"\nDecision Tree Feature Importance:")
    for i, importance in enumerate(dt_meta.feature_importances_):
        print(f"  {model_names[i]}: {importance:.4f}")

    return dt_meta

def train_logistic_regression_keras(X_train, y_train, X_val=None, y_val=None):
    # Create logistic regression model (single dense layer with sigmoid activation)
    model = Sequential([
        Dense(1, activation='sigmoid', input_shape=(3,), kernel_regularizer=tf.keras.regularizers.l2(0.001), name='logistic_layer')
    ])

    # Compile model
    model.compile(
        optimizer=Adam(learning_rate=0.01),
        loss='binary_crossentropy',
        metrics=['accuracy', 'precision', 'recall']
    )

    # Setup callbacks
    callbacks = [
        EarlyStopping(
            monitor='val_loss' if X_val is not None else 'loss',
            patience=50,
            restore_best_weights=True,
            verbose=0
        )
    ]

    # Prepare validation data
    validation_data = None
    if X_val is not None and y_val is not None:
        validation_data = (X_val, y_val)
    else:
        # Use 20% of training data for validation
        val_split = 0.2

    # Train model
    history = model.fit(
        X_train, y_train,
        epochs=200,
        batch_size=32,
        validation_split=0.2 if validation_data is None else None,
        validation_data=validation_data,
        callbacks=callbacks,
        verbose=0
    )

    # Print learned weights and bias
    weights, bias = model.get_layer('logistic_layer').get_weights()
    model_names = ["URLNet", "HTMLPhish", "WebPhish"]
    print(f"\nLogistic Regression (Keras) Coefficients:")
    for i, coef in enumerate(weights.flatten()):
        print(f"  {model_names[i]}: {coef:.4f}")
    print(f"  Bias: {bias[0]:.4f}")
    print(f"  Training epochs: {len(history.history['loss'])}")

    return model

def train_neural_network_keras(X_train, y_train, X_val=None, y_val=None):
    # Create neural network model
    model = Sequential([
        Dense(16, activation='relu', input_shape=(3,), kernel_regularizer=tf.keras.regularizers.l2(0.001), name='hidden_layer_1'),
        Dropout(0.3),

        Dense(8, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001), name='hidden_layer_2'),
        Dropout(0.2),

        Dense(1, activation='sigmoid', name='output_layer')
    ])

    # Compile model
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy', 'precision', 'recall']
    )

    # Setup callbacks
    callbacks = [
        EarlyStopping(
            monitor='val_loss' if X_val is not None else 'loss',
            patience=20,
            restore_best_weights=True,
            verbose=0
        )
    ]

    # Prepare validation data
    validation_data = None
    if X_val is not None and y_val is not None:
        validation_data = (X_val, y_val)

    # Train model
    history = model.fit(
        X_train, y_train,
        epochs=200,
        batch_size=32,
        validation_split=0.2 if validation_data is None else None,
        validation_data=validation_data,
        callbacks=callbacks,
        verbose=0
    )

    print(f"\nNeural Network (Keras) Architecture:")
    print(f"  Input Layer: 3 neurons")
    print(f"  Hidden Layer 1: 16 neurons (ReLU + Dropout 0.3)")
    print(f"  Hidden Layer 2: 8 neurons (ReLU + Dropout 0.2)")
    print(f"  Output Layer: 1 neuron (Sigmoid)")
    print(f"  Training epochs: {len(history.history['loss'])}")
    print(f"  Total parameters: {model.count_params()}")

    return model

In [ ]:
# --- Prediction Functions ---
def predict_with_ensemble(models_dict, predictions):
    predictions = np.array(predictions).astype(np.float32)
    results = {}

    # Basic ensemble methods
    results["majority_voting"] = majority_voting(predictions)
    results["mean_voting"] = mean_voting(predictions)
    results["most_certain"] = most_certain(predictions)

    # Meta-model methods
    if "decision_tree" in models_dict:
        results["decision_tree"] = models_dict["decision_tree"].predict(predictions)

    if "logistic_regression" in models_dict:
        lr_probs = models_dict["logistic_regression"].predict(predictions, verbose=0)
        results["logistic_regression"] = (lr_probs >= 0.5).astype(int).flatten()

    if "neural_network" in models_dict:
        nn_probs = models_dict["neural_network"].predict(predictions, verbose=0)
        results["neural_network"] = (nn_probs >= 0.5).astype(int).flatten()

    return results

def get_prediction_probabilities(models_dict, predictions):
    predictions = np.array(predictions).astype(np.float32)
    probs = {}

    if "logistic_regression" in models_dict:
        probs["logistic_regression"] = models_dict["logistic_regression"].predict(predictions, verbose=0).flatten()

    if "neural_network" in models_dict:
        probs["neural_network"] = models_dict["neural_network"].predict(predictions, verbose=0).flatten()

    return probs

In [ ]:
# --- Evaluation Function ---
def evaluate_predictions(y_true, y_pred, method_name):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"\n{method_name} Results:")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall   : {rec:.4f}")
    print(f"  F1-score : {f1:.4f}")

    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}

In [ ]:
def plot_all_stacking_methods(y_true, test_predictions, models_dict, output_dir="plots"):
    import os
    os.makedirs(output_dir, exist_ok=True)

    for method, preds in test_predictions.items():
        method_name = method.replace('_', ' ').title()
        print(f"\nGenerating plots for: {method_name}")

        # Determine probabilities if available
        if method in ["logistic_regression", "neural_network"]:
            probs = get_prediction_probabilities(models_dict, X_test)[method]
        else:
            # For methods without probabilities, use predicted labels as proxy
            probs = preds.astype(float)

        # Confusion Matrix
        cm = confusion_matrix(y_true, preds)
        plt.figure(figsize=(5,4))
        sns.heatmap(cm, annot=True, fmt='d', cmap="Blues", cbar=False,
                    xticklabels=["Legitimate", "Phishing"],
                    yticklabels=["Legitimate", "Phishing"])
        plt.title(f"{method_name} - Confusion Matrix")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.tight_layout()
        plt.savefig(f"{output_dir}/{method}_confusion_matrix.png")
        plt.close()

        # Precision-Recall Curve
        precision, recall, thresholds = precision_recall_curve(y_true, probs)
        plt.figure()
        plt.plot(recall, precision, marker='.', label=method_name)
        plt.title(f"{method_name} - Precision-Recall Curve")
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.legend()
        plt.tight_layout()
        plt.savefig(f"{output_dir}/{method}_precision_recall.png")
        plt.close()

        # Precision vs Confidence
        plt.figure()
        plt.plot(thresholds, precision[:-1], marker='.')
        plt.title(f"{method_name} - Precision vs Confidence")
        plt.xlabel("Confidence Threshold")
        plt.ylabel("Precision")
        plt.tight_layout()
        plt.savefig(f"{output_dir}/{method}_precision_confidence.png")
        plt.close()

        # Recall vs Confidence
        plt.figure()
        plt.plot(thresholds, recall[:-1], marker='.')
        plt.title(f"{method_name} - Recall vs Confidence")
        plt.xlabel("Confidence Threshold")
        plt.ylabel("Recall")
        plt.tight_layout()
        plt.savefig(f"{output_dir}/{method}_recall_confidence.png")
        plt.close()

        # F1 vs Confidence
        f1_scores = [f1_score(y_true, (probs >= t).astype(int)) for t in thresholds]
        plt.figure()
        plt.plot(thresholds, f1_scores, marker='.')
        plt.title(f"{method_name} - F1 vs Confidence")
        plt.xlabel("Confidence Threshold")
        plt.ylabel("F1-score")
        plt.tight_layout()
        plt.savefig(f"{output_dir}/{method}_f1_confidence.png")
        plt.close()

        # ROC Curve
        fpr, tpr, _ = roc_curve(y_true, probs)
        auc_score = roc_auc_score(y_true, probs)
        plt.figure()
        plt.plot(fpr, tpr, label=f"AUC = {auc_score:.4f}")
        plt.plot([0,1],[0,1], linestyle="--", color="gray")
        plt.title(f"{method_name} - ROC Curve (AUC={auc_score:.4f})")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.legend()
        plt.tight_layout()
        plt.savefig(f"{output_dir}/{method}_roc_auc.png")
        plt.close()

        print(f"Plots saved for {method_name} in '{output_dir}/'")

In [ ]:
def main():
    print("="*50)
    print("TRAINING META-MODELS")
    print("="*50)

    models = {}

    print("\n1. Training Decision Tree Meta-Model...")
    models["decision_tree"] = train_decision_tree_meta_model(X_train, y_train)
    print("Decision Tree trained successfully")

    print("\n2. Training Logistic Regression Meta-Model...")
    models["logistic_regression"] = train_logistic_regression_keras(X_train, y_train)
    print("Logistic Regression trained successfully")

    print("\n3. Training Neural Network Meta-Model...")
    models["neural_network"] = train_neural_network_keras(X_train, y_train)
    print("Neural Network trained successfully")

    print("\n" + "="*50)
    print("EVALUATION ON TEST SET")
    print("="*50)

    # Get predictions for all stacking/ensemble methods
    test_predictions = predict_with_ensemble(models, X_test)
    test_results = {}

    for method, preds in test_predictions.items():
        metrics = evaluate_predictions(y_test, preds, f"{method.replace('_', ' ').title()} (Test)")
        test_results[method] = metrics

    # Summary table with ROC-AUC
    from sklearn.metrics import roc_auc_score
    print("\n" + "="*60)
    print("SUMMARY COMPARISON (TEST SET)")
    print("="*60)
    print(f"{'Method':<25} {'Accuracy':<10} {'Precision':<10} {'Recall':<10} {'F1-Score':<10} {'ROC-AUC':<10}")
    print("-" * 75)

    for method, preds in test_predictions.items():
        metrics = test_results[method]

        # Compute ROC-AUC if probabilities available
        if method in ["logistic_regression", "neural_network"]:
            probs = get_prediction_probabilities(models, X_test)[method]
            roc_auc = roc_auc_score(y_test, probs)
        else:
            # Use predicted labels as proxy for ROC-AUC
            roc_auc = roc_auc_score(y_test, preds)

        print(f"{method.replace('_', ' ').title():<25} "
              f"{metrics['accuracy']:<10.4f} "
              f"{metrics['precision']:<10.4f} "
              f"{metrics['recall']:<10.4f} "
              f"{metrics['f1']:<10.4f} "
              f"{roc_auc:<10.4f}")

    # Best method based on F1-score
    best_method = max(test_results.items(), key=lambda x: x[1]['f1'])
    print(f"\nBest performing method: {best_method[0].replace('_', ' ').title()} "
          f"(F1-Score: {best_method[1]['f1']:.4f})")

    # Return models, test results, and predictions for plotting
    return models, test_results, test_predictions

In [ ]:
# Run the main function
if __name__ == "__main__":
    trained_models, test_results, test_predictions = main()

    plot_all_stacking_methods(y_test, test_predictions, trained_models, output_dir="plots")

TRAINING META-MODELS

1. Training Decision Tree Meta-Model...

Decision Tree Feature Importance:
  URLNet: 0.0420
  HTMLPhish: 0.0139
  WebPhish: 0.9441
Decision Tree trained successfully

2. Training Logistic Regression Meta-Model...

Logistic Regression (Keras) Coefficients:
  URLNet: 3.1921
  HTMLPhish: 2.6826
  WebPhish: 3.0238
  Bias: -4.5912
  Training epochs: 58
Logistic Regression trained successfully

3. Training Neural Network Meta-Model...

Neural Network (Keras) Architecture:
  Input Layer: 3 neurons
  Hidden Layer 1: 16 neurons (ReLU + Dropout 0.3)
  Hidden Layer 2: 8 neurons (ReLU + Dropout 0.2)
  Output Layer: 1 neuron (Sigmoid)
  Training epochs: 138
  Total parameters: 209
Neural Network trained successfully

EVALUATION ON TEST SET

Majority Voting (Test) Results:
  Accuracy : 0.9869
  Precision: 0.9886
  Recall   : 0.9851
  F1-score : 0.9869

Mean Voting (Test) Results:
  Accuracy : 0.9874
  Precision: 0.9893
  Recall   : 0.9855
  F1-score : 0.9874

Most Certain (Test

In [ ]:
def predict_single_sample(models_dict, urlnet_prob, htmlphish_prob, webphish_prob, verbose=True):
    """
    Make predictions for a single sample using all ensemble methods

    Args:
        models_dict: Dictionary containing trained meta-models
        urlnet_prob: URLNet prediction probability (0.0 to 1.0)
        htmlphish_prob: HTMLPhish prediction probability (0.0 to 1.0)
        webphish_prob: WebPhish prediction probability (0.0 to 1.0)
        verbose: Whether to print detailed results

    Returns:
        Dictionary with predictions from all ensemble methods
    """
    # Input validation
    def validate_probability(prob, name):
        if not isinstance(prob, (int, float)):
            raise ValueError(f"{name} must be a number between 0 and 1")
        if not (0.0 <= prob <= 1.0):
            raise ValueError(f"{name} must be between 0.0 and 1.0, got {prob}")

    validate_probability(urlnet_prob, "URLNet probability")
    validate_probability(htmlphish_prob, "HTMLPhish probability")
    validate_probability(webphish_prob, "WebPhish probability")

    # Create prediction array
    predictions = np.array([[urlnet_prob, htmlphish_prob, webphish_prob]], dtype=np.float32)

    if verbose:
        print("="*60)
        print("SINGLE SAMPLE PREDICTION")
        print("="*60)
        print(f"Input Probabilities:")
        print(f"  URLNet     : {urlnet_prob:.4f}")
        print(f"  HTMLPhish  : {htmlphish_prob:.4f}")
        print(f"  WebPhish   : {webphish_prob:.4f}")
        print("\n" + "-"*60)
        print("ENSEMBLE PREDICTIONS:")
        print("-"*60)

    results = {}

    # Basic ensemble methods
    majority_pred = majority_voting(predictions)[0]
    mean_pred = mean_voting(predictions)[0]
    certain_pred = most_certain(predictions)[0]

    results["majority_voting"] = majority_pred
    results["mean_voting"] = mean_pred
    results["most_certain"] = certain_pred

    if verbose:
        print(f"Majority Voting    : {'PHISHING' if majority_pred == 1 else 'LEGITIMATE'} ({majority_pred})")
        print(f"Mean Voting        : {'PHISHING' if mean_pred == 1 else 'LEGITIMATE'} ({mean_pred})")
        print(f"Most Certain       : {'PHISHING' if certain_pred == 1 else 'LEGITIMATE'} ({certain_pred})")

    # Meta-model methods
    if "decision_tree" in models_dict:
        dt_pred = models_dict["decision_tree"].predict(predictions)[0]
        results["decision_tree"] = dt_pred
        if verbose:
            print(f"Decision Tree      : {'PHISHING' if dt_pred == 1 else 'LEGITIMATE'} ({dt_pred})")

    if "logistic_regression" in models_dict:
        lr_prob = models_dict["logistic_regression"].predict(predictions, verbose=0)[0][0]
        lr_pred = 1 if lr_prob >= 0.5 else 0
        results["logistic_regression"] = lr_pred
        results["logistic_regression_prob"] = lr_prob
        if verbose:
            print(f"Logistic Regression: {'PHISHING' if lr_pred == 1 else 'LEGITIMATE'} ({lr_pred}) [prob: {lr_prob:.4f}]")

    if "neural_network" in models_dict:
        nn_prob = models_dict["neural_network"].predict(predictions, verbose=0)[0][0]
        nn_pred = 1 if nn_prob >= 0.5 else 0
        results["neural_network"] = nn_pred
        results["neural_network_prob"] = nn_prob
        if verbose:
            print(f"Neural Network     : {'PHISHING' if nn_pred == 1 else 'LEGITIMATE'} ({nn_pred}) [prob: {nn_prob:.4f}]")

    # Consensus analysis
    if verbose:
        predictions_only = [v for k, v in results.items() if not k.endswith('_prob')]
        phishing_votes = sum(predictions_only)
        total_votes = len(predictions_only)

        print("\n" + "-"*60)
        print("CONSENSUS ANALYSIS:")
        print("-"*60)
        print(f"Phishing votes: {phishing_votes}/{total_votes}")

        if phishing_votes > total_votes / 2:
            print(f"Overall Result: PHISHING (confidence: {phishing_votes/total_votes:.1%})")
        else:
            print(f"Overall Result: LEGITIMATE (confidence: {(total_votes-phishing_votes)/total_votes:.1%})")
        print("="*60)

    return results


def interactive_prediction(models_dict):
    """
    Interactive function to get user input and make predictions (one-time use)

    Args:
        models_dict: Dictionary containing trained meta-models
    """
    print("\n" + "="*70)
    print("INTERACTIVE PHISHING DETECTION ENSEMBLE")
    print("="*70)
    print("Enter confidence scores from each base model (0.0 to 1.0)")
    print("Higher values indicate higher confidence that the URL is PHISHING")
    print("="*70)

    try:
        print("\nEnter prediction probabilities:")

        # Get URLNet probability
        while True:
            try:
                urlnet_input = input("URLNet confidence (0.0-1.0): ").strip()
                urlnet_prob = float(urlnet_input)
                if 0.0 <= urlnet_prob <= 1.0:
                    break
                else:
                    print("Please enter a value between 0.0 and 1.0")
            except ValueError:
                print("Please enter a valid number")

        # Get HTMLPhish probability
        while True:
            try:
                htmlphish_input = input("HTMLPhish confidence (0.0-1.0): ").strip()
                htmlphish_prob = float(htmlphish_input)
                if 0.0 <= htmlphish_prob <= 1.0:
                    break
                else:
                    print("Please enter a value between 0.0 and 1.0")
            except ValueError:
                print("Please enter a valid number")

        # Get WebPhish probability
        while True:
            try:
                webphish_input = input("WebPhish confidence (0.0-1.0): ").strip()
                webphish_prob = float(webphish_input)
                if 0.0 <= webphish_prob <= 1.0:
                    break
                else:
                    print("Please enter a value between 0.0 and 1.0")
            except ValueError:
                print("Please enter a valid number")

        # Make prediction
        results = predict_single_sample(models_dict, urlnet_prob, htmlphish_prob, webphish_prob)
        return results

    except KeyboardInterrupt:
        print("\n\nExiting...")
        return None
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

In [ ]:
# For interactive single predictions
interactive_prediction(trained_models)


INTERACTIVE PHISHING DETECTION ENSEMBLE
Enter confidence scores from each base model (0.0 to 1.0)
Higher values indicate higher confidence that the URL is PHISHING

Enter prediction probabilities:
URLNet confidence (0.0-1.0): 0.8
HTMLPhish confidence (0.0-1.0): 0.8
WebPhish confidence (0.0-1.0): 0.8
SINGLE SAMPLE PREDICTION
Input Probabilities:
  URLNet     : 0.8000
  HTMLPhish  : 0.8000
  WebPhish   : 0.8000

------------------------------------------------------------
ENSEMBLE PREDICTIONS:
------------------------------------------------------------
Majority Voting    : PHISHING (1)
Mean Voting        : PHISHING (1)
Most Certain       : PHISHING (1)
Decision Tree      : PHISHING (1)
Logistic Regression: PHISHING (1) [prob: 0.9261]
Neural Network     : PHISHING (1) [prob: 0.9892]

------------------------------------------------------------
CONSENSUS ANALYSIS:
------------------------------------------------------------
Phishing votes: 6/6
Overall Result: PHISHING (confidence: 100.0%

{'majority_voting': np.int64(1),
 'mean_voting': np.int64(1),
 'most_certain': np.int64(1),
 'decision_tree': np.int64(1),
 'logistic_regression': 1,
 'logistic_regression_prob': np.float32(0.9260515),
 'neural_network': 1,
 'neural_network_prob': np.float32(0.9891897)}

In [ ]:
!zip -r /content/file.zip /content/plots

  adding: content/plots/ (stored 0%)
  adding: content/plots/Neural_Network_precision_recall.png (deflated 15%)
  adding: content/plots/neural_network_f1_confidence.png (deflated 13%)
  adding: content/plots/neural_network_recall_confidence.png (deflated 12%)
  adding: content/plots/most_certain_recall_confidence.png (deflated 15%)
  adding: content/plots/Logistic_Regression_precision_confidence.png (deflated 13%)
  adding: content/plots/majority_voting_roc_auc.png (deflated 12%)
  adding: content/plots/most_certain_precision_confidence.png (deflated 11%)
  adding: content/plots/logistic_regression_recall_confidence.png (deflated 12%)
  adding: content/plots/most_certain_precision_recall.png (deflated 13%)
  adding: content/plots/logistic_regression_precision_confidence.png (deflated 13%)
  adding: content/plots/majority_voting_precision_confidence.png (deflated 11%)
  adding: content/plots/Logistic_Regression_confusion_matrix.png (deflated 12%)
  adding: content/plots/most_certain_con

In [ ]:
from google.colab import files
files.download("/content/file.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>